# 🛒 Market Basket Analyzer (Apriori Algorithm)

A comprehensive market basket analysis pipeline that discovers associations between
items in transactional data and surfaces actionable business insights.

**Contents**
1. Configuration & Logging Setup
2. Importing Libraries
3. Data Loading & Validation
4. Data Cleaning & Exploration
5. Exploratory Data Analysis (EDA)
6. Enhanced Data Profiling
7. Data Transformation
8. Apriori Algorithm & Association Rules
9. Extended Metrics (Leverage, Conviction, Kulczynski)
10. Statistical Testing (Chi-Square)
11. Comprehensive Visualisations
12. Business Intelligence Insights
13. Key Findings Summary


## 1. Configuration & Logging Setup

In [ ]:
import yaml
import logging
import os
import warnings
warnings.filterwarnings('ignore')

# ── Load configuration ──────────────────────────────────────────────────────
CONFIG_FILE = 'config.yaml'
if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, 'r') as f:
        cfg = yaml.safe_load(f)
    print(f'✅ Configuration loaded from {CONFIG_FILE}')
else:
    # Fallback defaults if config.yaml is missing
    cfg = {
        'data': {'filepath': 'Groceries_dataset.csv',
                 'member_col': 'Member_number',
                 'date_col': 'Date',
                 'item_col': 'itemDescription'},
        'apriori': {'min_support': 0.001, 'metric': 'confidence',
                    'min_threshold': 0.05, 'max_len': None},
        'visualisation': {'dpi': 100, 'palette': 'viridis',
                          'top_n_rules': 20, 'network_top_n': 50},
        'statistics': {'alpha': 0.05},
        'business': {'avg_transaction_value': 25.0,
                     'recommendation_conversion_rate': 0.10},
        'logging': {'level': 'INFO', 'logfile': 'market_basket_analysis.log'},
        'random_seed': 42,
    }
    print('⚠️  config.yaml not found – using default parameters')

# ── Configure logging ───────────────────────────────────────────────────────
log_level = getattr(logging, cfg['logging']['level'], logging.INFO)
log_handlers = [logging.StreamHandler()]
if cfg['logging'].get('logfile'):
    log_handlers.append(logging.FileHandler(cfg['logging']['logfile']))

logging.basicConfig(
    level=log_level,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=log_handlers
)
logger = logging.getLogger('MarketBasketAnalyzer')
logger.info('Market Basket Analyzer started')
print('✅ Logging configured')


## 2. Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chi2_contingency
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Reproducibility
SEED = cfg['random_seed']
np.random.seed(SEED)

# Plot style
plt.rcParams.update({'figure.dpi': cfg['visualisation']['dpi'],
                     'figure.figsize': (10, 6)})
sns.set_theme(style='whitegrid')

logger.info('All libraries imported successfully')
print('✅ Libraries imported')


## 3. Data Loading & Validation

We load the dataset and immediately validate it for common data-quality issues.


In [ ]:
# ── Helper: validate dataset ──────────────────────────────────────────────
def validate_dataframe(df, member_col, date_col, item_col):
    """Run basic data-quality checks and log warnings."""
    issues = []
    required_cols = [member_col, date_col, item_col]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f'Missing required columns: {missing_cols}')

    null_counts = df[required_cols].isnull().sum()
    for col, cnt in null_counts.items():
        if cnt > 0:
            issues.append(f'  ⚠️  Column "{col}" has {cnt} null values ({cnt/len(df)*100:.1f}%)')

    dup_count = df.duplicated().sum()
    if dup_count > 0:
        issues.append(f'  ⚠️  {dup_count} duplicate rows detected')

    if issues:
        print('Data quality warnings:')
        for issue in issues:
            print(issue)
            logger.warning(issue)
    else:
        print('  ✅ No data-quality issues detected')
    return issues


# ── Load dataset ──────────────────────────────────────────────────────────────
DATA_PATH = cfg['data']['filepath']
MEMBER_COL = cfg['data']['member_col']
DATE_COL   = cfg['data']['date_col']
ITEM_COL   = cfg['data']['item_col']

try:
    df_raw = pd.read_csv(DATA_PATH)
    logger.info(f'Dataset loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
    print(f'✅ Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
except FileNotFoundError:
    logger.error(f'Dataset file not found: {DATA_PATH}')
    raise FileNotFoundError(
        f'Dataset not found at "{DATA_PATH}". '
        'Please update the "data.filepath" key in config.yaml '
        'or place the CSV in the same directory as the notebook.'
    )

print('\nData quality check:')
_ = validate_dataframe(df_raw, MEMBER_COL, DATE_COL, ITEM_COL)
df_raw.head()


## 4. Data Cleaning & Exploration

In [ ]:
# Drop nulls and duplicates
df = df_raw.dropna(subset=[MEMBER_COL, DATE_COL, ITEM_COL]).drop_duplicates()
df[ITEM_COL] = df[ITEM_COL].str.strip()

logger.info(f'After cleaning: {len(df):,} rows')
print(f'Rows after cleaning: {len(df):,}')
print(f'Shape: {df.shape}')
df.info()


In [ ]:
print('Null counts per column:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')


In [ ]:
# Group items purchased together by the same customer on the same date
item_list = (
    df.groupby([MEMBER_COL, DATE_COL])[ITEM_COL]
    .apply(lambda x: ', '.join(x))
    .reset_index(name='items')
)
item_list.head(15)


In [ ]:
print(f'Unique customers : {df[MEMBER_COL].nunique():,}')
print(f'Unique dates     : {df[DATE_COL].nunique():,}')
print(f'Unique items     : {df[ITEM_COL].nunique():,}')
print(f'Total rows       : {len(df):,}')
df[ITEM_COL].value_counts().head(10)


## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Top 15 most purchased items
top_items = df[ITEM_COL].value_counts().head(15)
fig, ax = plt.subplots(figsize=(12, 5))
top_items.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 15 Most Purchased Items', fontsize=14)
ax.set_xlabel('Item')
ax.set_ylabel('Purchase Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# 15 least purchased items
least_items = df[ITEM_COL].value_counts().tail(15).sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
least_items.plot(kind='barh', ax=ax, color='salmon')
ax.set_title('15 Least Purchased Items', fontsize=14)
ax.set_xlabel('Purchase Count')
plt.tight_layout()
plt.show()


In [ ]:
# Transactions per customer
transactions_per_customer = df.groupby(MEMBER_COL)[DATE_COL].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
transactions_per_customer.hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Transactions per Customer')
axes[0].set_xlabel('Number of Transactions')
axes[0].set_ylabel('Number of Customers')

axes[1].boxplot(transactions_per_customer, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Customer Transaction Distribution (Boxplot)')
axes[1].set_ylabel('Transactions')
plt.tight_layout()
plt.show()

print(transactions_per_customer.describe())


In [ ]:
# Top 10 most active customers
top_customers = transactions_per_customer.sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10, 5))
top_customers.plot(kind='bar', ax=ax, color='darkorange')
ax.set_title('Top 10 Most Active Customers')
ax.set_xlabel('Customer ID')
ax.set_ylabel('Number of Transactions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Customer segmentation
customer_segments = pd.cut(
    transactions_per_customer,
    bins=[0, 5, 10, transactions_per_customer.max()],
    labels=['Low (1-5)', 'Medium (6-10)', 'High (>10)']
)
segment_counts = customer_segments.value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
segment_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[0].set_title('Customer Segmentation by Transaction Frequency')
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Number of Customers')
plt.setp(axes[0].get_xticklabels(), rotation=0)

axes[1].pie(segment_counts, labels=segment_counts.index,
            autopct='%1.1f%%', startangle=90,
            colors=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('Customer Segment Distribution')
plt.tight_layout()
plt.show()
print(segment_counts)


## 6. Enhanced Data Profiling

This section adds sparsity metrics, basket-size statistics,
and outlier detection to help understand the shape of the data.


In [ ]:
basket_size = df.groupby([MEMBER_COL, DATE_COL])[ITEM_COL].count()
n_transactions = len(basket_size)
n_unique_items  = df[ITEM_COL].nunique()

# Sparsity: how many (transaction, item) pairs are empty vs. total possible
total_possible = n_transactions * n_unique_items
actual_pairs   = len(df)
sparsity       = 1 - (actual_pairs / total_possible)

print('=' * 50)
print('  DATA PROFILING SUMMARY')
print('=' * 50)
print(f'  Total transactions  : {n_transactions:,}')
print(f'  Unique items        : {n_unique_items:,}')
print(f'  Unique customers    : {df[MEMBER_COL].nunique():,}')
print(f'  Unique dates        : {df[DATE_COL].nunique():,}')
print(f'  Dataset sparsity    : {sparsity:.4%}')
print(f'  Avg basket size     : {basket_size.mean():.2f} items')
print(f'  Median basket size  : {basket_size.median():.1f} items')
print(f'  Max basket size     : {basket_size.max()} items')
print(f'  Min basket size     : {basket_size.min()} items')
print('=' * 50)
logger.info(f'Dataset sparsity: {sparsity:.4%}')


In [ ]:
# Basket size distribution with outlier detection
Q1 = basket_size.quantile(0.25)
Q3 = basket_size.quantile(0.75)
IQR = Q3 - Q1
outlier_mask = (basket_size < Q1 - 1.5 * IQR) | (basket_size > Q3 + 1.5 * IQR)
n_outliers = outlier_mask.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
basket_size.hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Basket Size Distribution')
axes[0].set_xlabel('Items per Transaction')
axes[0].set_ylabel('Frequency')

# Boxplot highlighting outliers
bp = axes[1].boxplot(basket_size, patch_artist=True,
                     boxprops=dict(facecolor='lightblue'),
                     flierprops=dict(marker='o', color='red', alpha=0.5))
axes[1].set_title(f'Basket Size Boxplot\n({n_outliers} outlier transactions)')
axes[1].set_ylabel('Items per Transaction')
plt.tight_layout()
plt.show()

print(f'Outlier transactions (basket size outside IQR range): {n_outliers}')


In [ ]:
# Item frequency distribution
item_freq = df[ITEM_COL].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
item_freq.hist(bins=50, ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Item Purchase Frequency Distribution')
ax.set_xlabel('Number of Purchases')
ax.set_ylabel('Number of Items')
ax.axvline(item_freq.mean(), color='red', linestyle='--',
           label=f'Mean = {item_freq.mean():.1f}')
ax.axvline(item_freq.median(), color='orange', linestyle='--',
           label=f'Median = {item_freq.median():.1f}')
ax.legend()
plt.tight_layout()
plt.show()


## 7. Data Transformation

In [ ]:
# Build transaction lists
transactions = (
    df.groupby([MEMBER_COL, DATE_COL])[ITEM_COL]
    .apply(list)
    .tolist()
)
print(f'Total transactions: {len(transactions):,}')
print('Sample (first 3):', transactions[:3])


In [ ]:
# One-hot encode transactions
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
# Ensure column names are plain Python strings (required for mlxtend >=0.24)
basket_df = pd.DataFrame(te_ary, columns=[str(col) for col in te.columns_])
print(f'One-hot encoded shape: {basket_df.shape}')
basket_df.head(3)


## 8. Apriori Algorithm & Association Rules

**Algorithm overview**

The Apriori algorithm works in two stages:
1. **Frequent Itemset Mining** – finds all itemsets whose *support* exceeds `min_support`
   using a breadth-first, prune-early strategy.
2. **Rule Generation** – derives if-then rules from the frequent itemsets and keeps only
   rules whose chosen metric (e.g. *confidence*) exceeds `min_threshold`.

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| Support | P(A ∩ B) | How often A and B appear together |
| Confidence | P(B\|A) | How often B appears when A is bought |
| Lift | P(A ∩ B) / (P(A)·P(B)) | Lift > 1 means positive association |


In [ ]:
MIN_SUPPORT   = cfg['apriori']['min_support']
METRIC        = cfg['apriori']['metric']
MIN_THRESHOLD = cfg['apriori']['min_threshold']
MAX_LEN       = cfg['apriori']['max_len']

logger.info(f'Running Apriori: min_support={MIN_SUPPORT}, metric={METRIC}, '
            f'min_threshold={MIN_THRESHOLD}')

# Step 1 – frequent itemsets
frequent_itemsets = apriori(
    basket_df,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_LEN,
)
print(f'Frequent itemsets found : {len(frequent_itemsets):,}')

# Step 2 – association rules
# num_itemsets is the total number of transactions (required in mlxtend >=0.24)
basket_rules = association_rules(
    frequent_itemsets,
    num_itemsets=len(basket_df),
    metric=METRIC,
    min_threshold=MIN_THRESHOLD,
)
print(f'Association rules found : {len(basket_rules):,}')
logger.info(f'Rules generated: {len(basket_rules)}')
basket_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)


In [ ]:
print('Association rules summary:')
print(basket_rules[['support', 'confidence', 'lift']].describe().round(4))


## 9. Extended Metrics

Beyond the standard support/confidence/lift, we compute three additional metrics
that each capture a different facet of association strength:

| Metric | Formula | What it tells you |
|--------|---------|-------------------|
| **Leverage** | P(A∩B) − P(A)·P(B) | Deviation from independence; > 0 means positive association |
| **Conviction** | (1 − P(B)) / (1 − Confidence) | Higher values mean the rule is less likely due to chance |
| **Kulczynski** | ½ · (P(A\|B) + P(B\|A)) | Symmetric measure of co-occurrence strength (0–1) |


In [ ]:
# mlxtend >=0.24 already computes leverage, conviction, and kulczynski.
# The helper functions below document the formulas for reference and
# serve as a fallback for older mlxtend versions.

def compute_conviction(confidence, consequent_support):
    """Conviction = (1 - P(consequent)) / (1 - confidence). Returns inf when confidence == 1."""
    denom = 1 - confidence
    with np.errstate(divide='ignore', invalid='ignore'):
        conv = np.where(denom == 0, np.inf, (1 - consequent_support) / denom)
    return conv

def compute_kulczynski(antecedent_support, consequent_support, support):
    """Kulczynski = 0.5 * (P(A∩B)/P(A) + P(A∩B)/P(B))."""
    with np.errstate(divide='ignore', invalid='ignore'):
        k = 0.5 * ((support / antecedent_support) + (support / consequent_support))
    return k


# Use mlxtend-computed values if present; otherwise compute manually
basket_rules = basket_rules.copy()
if 'conviction' not in basket_rules.columns:
    basket_rules['conviction'] = compute_conviction(
        basket_rules['confidence'].values,
        basket_rules['consequent support'].values,
    )
if 'kulczynski' not in basket_rules.columns:
    basket_rules['kulczynski'] = compute_kulczynski(
        basket_rules['antecedent support'].values,
        basket_rules['consequent support'].values,
        basket_rules['support'].values,
    )

cols = ['antecedents', 'consequents', 'support', 'confidence',
        'lift', 'leverage', 'conviction', 'kulczynski']
print('Extended metrics (top 10 by lift):')
basket_rules.sort_values('lift', ascending=False)[cols].head(10)


## 10. Statistical Testing – Chi-Square Independence

A high lift value alone does not guarantee that an association is *statistically significant*.
For each rule A ⇒ B we run a **chi-square test of independence** on the 2×2 contingency table
of (A present/absent) × (B present/absent). Rules where *p-value < α* are flagged as significant.


In [ ]:
ALPHA = cfg['statistics']['alpha']
N     = len(basket_df)   # total number of transactions

def chi_square_rule(row, basket_df):
    """Compute chi-square test for a single association rule."""
    ant_items = list(row['antecedents'])
    con_items = list(row['consequents'])

    ant_present = basket_df[ant_items].all(axis=1)
    con_present = basket_df[con_items].all(axis=1)

    # 2×2 contingency table
    n11 = (ant_present & con_present).sum()    # both present
    n10 = (ant_present & ~con_present).sum()   # ant only
    n01 = (~ant_present & con_present).sum()   # con only
    n00 = (~ant_present & ~con_present).sum()  # neither

    contingency = [[n11, n10], [n01, n00]]
    chi2, p_val, _, _ = chi2_contingency(contingency, correction=False)
    return pd.Series({'chi2': chi2, 'p_value': p_val})


print('Running chi-square tests … (this may take a moment)')
chi_results = basket_rules.apply(chi_square_rule, axis=1, basket_df=basket_df)
basket_rules[['chi2', 'p_value']] = chi_results
basket_rules['significant'] = basket_rules['p_value'] < ALPHA

n_sig = basket_rules['significant'].sum()
print(f'\nSignificant rules (p < {ALPHA}): {n_sig} / {len(basket_rules)}')
logger.info(f'Chi-square: {n_sig}/{len(basket_rules)} rules are statistically significant')

# Show top significant rules
sig_rules = basket_rules[basket_rules['significant']].sort_values('lift', ascending=False)
display_cols = ['antecedents', 'consequents', 'support', 'confidence',
                'lift', 'kulczynski', 'chi2', 'p_value']
sig_rules[display_cols].head(10)


## 11. Comprehensive Visualisations

### 11a. Support vs Confidence vs Lift (3-D Bubble Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(
    basket_rules['support'],
    basket_rules['confidence'],
    s=basket_rules['lift'] * 30,
    c=basket_rules['lift'],
    cmap='viridis',
    alpha=0.6,
    edgecolors='none',
)
plt.colorbar(sc, ax=ax, label='Lift')
ax.set_xlabel('Support')
ax.set_ylabel('Confidence')
ax.set_title('Association Rules: Support vs Confidence\n(bubble size & colour = Lift)')
plt.tight_layout()
plt.show()


### 11b. Metric Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
metrics_to_plot = ['support', 'confidence', 'lift', 'leverage', 'conviction', 'kulczynski']
colours = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6', '#e74c3c', '#1abc9c']

for ax, metric, colour in zip(axes.flatten(), metrics_to_plot, colours):
    data = basket_rules[metric].replace([np.inf, -np.inf], np.nan).dropna()
    ax.hist(data, bins=30, color=colour, edgecolor='white', alpha=0.85)
    ax.set_title(f'{metric.capitalize()} Distribution')
    ax.set_xlabel(metric.capitalize())
    ax.set_ylabel('Count')
    ax.axvline(data.mean(), color='black', linestyle='--',
               linewidth=1, label=f'Mean={data.mean():.3f}')
    ax.legend(fontsize=8)

plt.suptitle('Association Rule Metric Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


### 11c. Top Rules – Multi-Metric Bar Chart

In [ ]:
TOP_N = cfg['visualisation']['top_n_rules']
top_rules = basket_rules.sort_values('lift', ascending=False).head(TOP_N).copy()
top_rules['rule_label'] = [
    f"{', '.join(list(a))} → {', '.join(list(c))}"
    for a, c in zip(top_rules['antecedents'], top_rules['consequents'])
]

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
for ax, metric, colour in zip(axes, ['support', 'confidence', 'lift'],
                               ['#3498db', '#2ecc71', '#e67e22']):
    ax.barh(top_rules['rule_label'], top_rules[metric], color=colour)
    ax.set_xlabel(metric.capitalize())
    ax.set_title(f'Top {TOP_N} Rules by {metric.capitalize()}')
    ax.invert_yaxis()

plt.suptitle(f'Top {TOP_N} Association Rules – Multi-Metric Comparison', fontsize=13)
plt.tight_layout()
plt.show()


### 11d. Item Co-occurrence Heatmap

In [ ]:
# Build co-occurrence matrix for top N items
TOP_ITEMS_HEATMAP = 20
top_item_names = df[ITEM_COL].value_counts().head(TOP_ITEMS_HEATMAP).index.tolist()
sub_basket = basket_df[top_item_names].astype(int)
cooc_matrix = sub_basket.T.dot(sub_basket)

# Mask the diagonal (self co-occurrence is trivial)
np.fill_diagonal(cooc_matrix.values, 0)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cooc_matrix,
    ax=ax,
    cmap='YlOrRd',
    annot=True,
    fmt='d',
    linewidths=0.5,
    cbar_kws={'label': 'Co-occurrence Count'},
)
ax.set_title(f'Item Co-occurrence Heatmap (Top {TOP_ITEMS_HEATMAP} Items)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


### 11e. Association Strength Heatmap (Confidence)

In [ ]:
# Pivot table of confidence for top items
TOP_ITEMS_ASSOC = 15
top_n_items = df[ITEM_COL].value_counts().head(TOP_ITEMS_ASSOC).index.tolist()

# Filter rules where both antecedent and consequent are a single top item
single_ant = basket_rules[
    basket_rules['antecedents'].apply(lambda x: len(x) == 1) &
    basket_rules['consequents'].apply(lambda x: len(x) == 1)
].copy()
single_ant['ant_item'] = single_ant['antecedents'].apply(lambda x: list(x)[0])
single_ant['con_item'] = single_ant['consequents'].apply(lambda x: list(x)[0])
filtered = single_ant[
    single_ant['ant_item'].isin(top_n_items) &
    single_ant['con_item'].isin(top_n_items)
]

if not filtered.empty:
    pivot = filtered.pivot_table(
        index='ant_item', columns='con_item', values='confidence', aggfunc='max'
    ).reindex(index=top_n_items, columns=top_n_items)

    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(pivot, ax=ax, cmap='Blues', annot=True, fmt='.2f',
                linewidths=0.5, cbar_kws={'label': 'Confidence'})
    ax.set_title(f'Association Confidence Heatmap (Top {TOP_ITEMS_ASSOC} Items)',
                 fontsize=13)
    ax.set_xlabel('Consequent')
    ax.set_ylabel('Antecedent')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No single-item rules found between the top items at the current thresholds.')


### 11f. Association Rules Network Graph

In [ ]:
NETWORK_TOP_N = cfg['visualisation']['network_top_n']
rules_net = basket_rules.sort_values('lift', ascending=False).head(NETWORK_TOP_N)

G = nx.DiGraph()
for _, row in rules_net.iterrows():
    for ant in row['antecedents']:
        for con in row['consequents']:
            G.add_edge(ant, con, weight=row['lift'],
                       confidence=row['confidence'])

# Node degree for sizing
node_sizes = [800 + 200 * G.degree(n) for n in G.nodes()]
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]

fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.6, seed=SEED)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes,
                       node_color='lightcoral', alpha=0.85, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
nx.draw_networkx_edges(G, pos, width=[w * 0.5 for w in edge_weights],
                       edge_color=edge_weights, edge_cmap=plt.cm.Blues,
                       arrows=True, arrowsize=15, ax=ax)
ax.set_title(f'Association Rules Network Graph (Top {NETWORK_TOP_N} by Lift)',
             fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()


### 11g. Rule Clustering (K-Means on Support / Confidence / Lift)

In [ ]:
features = basket_rules[['support', 'confidence', 'lift']].copy()
scaler   = StandardScaler()
features_scaled = scaler.fit_transform(features)

k = 5
kmeans   = KMeans(n_clusters=k, random_state=SEED, n_init='auto')
clusters = kmeans.fit_predict(features_scaled)
basket_rules['cluster'] = clusters

fig, ax = plt.subplots(figsize=(10, 7))
palette = sns.color_palette('tab10', k)
for cluster_id in range(k):
    mask = basket_rules['cluster'] == cluster_id
    ax.scatter(
        basket_rules.loc[mask, 'support'],
        basket_rules.loc[mask, 'confidence'],
        label=f'Cluster {cluster_id}',
        s=basket_rules.loc[mask, 'lift'] * 30,
        alpha=0.6,
        color=palette[cluster_id],
    )
ax.set_xlabel('Support')
ax.set_ylabel('Confidence')
ax.set_title('Association Rule Clusters (K-Means, k=5)')
ax.legend(title='Cluster')
plt.tight_layout()
plt.show()

# Cluster summary
cluster_summary = basket_rules.groupby('cluster')[['support','confidence','lift']].mean()
print('\nCluster centroids (mean metrics):')
print(cluster_summary.round(4))


### 11h. Parallel Coordinates Plot (Interactive)

In [ ]:
rules_sub = basket_rules.head(50).copy()
rules_sub['rule'] = [
    f"{', '.join(list(a))} → {', '.join(list(c))}"
    for a, c in zip(rules_sub['antecedents'], rules_sub['consequents'])
]
fig = px.parallel_coordinates(
    rules_sub,
    dimensions=['support', 'confidence', 'lift', 'leverage', 'kulczynski'],
    color='lift',
    color_continuous_scale=px.colors.sequential.Viridis,
    labels={
        'support': 'Support', 'confidence': 'Confidence',
        'lift': 'Lift', 'leverage': 'Leverage', 'kulczynski': 'Kulczynski',
    },
    title='Parallel Coordinates – Top 50 Association Rules (coloured by Lift)',
)
fig.show()


## 12. Business Intelligence Insights

This section translates the statistical findings into actionable business recommendations:
- **ROI / impact estimation** for the top rules
- **Cross-selling opportunity scores**
- **Bundle pricing recommendations**
- **Store layout optimisation hints**


In [ ]:
AVG_TXN_VALUE = cfg['business']['avg_transaction_value']
CONV_RATE     = cfg['business']['recommendation_conversion_rate']
TOTAL_TXN     = len(basket_df)

# ── ROI Estimation ────────────────────────────────────────────────────────────
# Estimated uplift transactions = antecedent transactions * confidence * conversion rate
bi_rules = basket_rules.sort_values('lift', ascending=False).head(20).copy()
bi_rules['ant_transactions'] = (bi_rules['antecedent support'] * TOTAL_TXN).round().astype(int)
bi_rules['estimated_upsell_txns'] = (
    bi_rules['ant_transactions'] * bi_rules['confidence'] * CONV_RATE
).round().astype(int)
bi_rules['estimated_revenue_impact'] = bi_rules['estimated_upsell_txns'] * AVG_TXN_VALUE

# Cross-selling opportunity score: lift × confidence
bi_rules['cross_sell_score'] = (bi_rules['lift'] * bi_rules['confidence']).round(4)

print(f'Assumptions:')
print(f'  Average transaction value : ${AVG_TXN_VALUE:.2f}')
print(f'  Recommendation conversion : {CONV_RATE*100:.0f}%')
print(f'  Total transactions        : {TOTAL_TXN:,}')
print()

bi_display = ['antecedents', 'consequents', 'lift', 'confidence',
              'ant_transactions', 'estimated_upsell_txns',
              'estimated_revenue_impact', 'cross_sell_score']
bi_rules[bi_display].head(10)


In [ ]:
# Visualise revenue impact
bi_rules['rule_label'] = [
    f"{', '.join(list(a))} → {', '.join(list(c))}"
    for a, c in zip(bi_rules['antecedents'], bi_rules['consequents'])
]
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

bi_sorted = bi_rules.sort_values('estimated_revenue_impact', ascending=True)
axes[0].barh(bi_sorted['rule_label'], bi_sorted['estimated_revenue_impact'],
             color='#27ae60')
axes[0].set_xlabel('Estimated Revenue Impact ($)')
axes[0].set_title('Top 20 Rules: Estimated Revenue Impact')

bi_sorted2 = bi_rules.sort_values('cross_sell_score', ascending=True)
axes[1].barh(bi_sorted2['rule_label'], bi_sorted2['cross_sell_score'],
             color='#8e44ad')
axes[1].set_xlabel('Cross-Sell Opportunity Score (Lift × Confidence)')
axes[1].set_title('Top 20 Rules: Cross-Selling Score')

plt.tight_layout()
plt.show()


In [ ]:
# Bundle recommendations (rules with high kulczynski)
bundle_candidates = (
    basket_rules[basket_rules['kulczynski'] >= 0.1]
    .sort_values('kulczynski', ascending=False)
    .head(10)
)
if not bundle_candidates.empty:
    print('📦 Bundle Pricing Candidates (high Kulczynski score):')
    for _, row in bundle_candidates.iterrows():
        ant = ', '.join(list(row['antecedents']))
        con = ', '.join(list(row['consequents']))
        print(f'  {ant} + {con}  '
              f'(Kulczynski={row["kulczynski"]:.3f}, '
              f'Lift={row["lift"]:.2f}, '
              f'Support={row["support"]:.4f})')
else:
    print('No bundle candidates found at the current Kulczynski threshold.')
    print('Try lowering the threshold or adjusting min_support in config.yaml.')


In [ ]:
# Store layout hints – items frequently bought together should be placed close to each other
layout_hints = (
    basket_rules.sort_values('confidence', ascending=False)
    .head(10)
)
print('🏪 Store Layout Optimisation Hints (highest confidence rules):')
for _, row in layout_hints.iterrows():
    ant = ', '.join(list(row['antecedents']))
    con = ', '.join(list(row['consequents']))
    print(f'  Place "{con}" near "{ant}"  '
          f'(Confidence={row["confidence"]:.3f}, Lift={row["lift"]:.2f})')


## 13. Key Findings Summary

In [ ]:
print('=' * 60)
print('  MARKET BASKET ANALYSIS – KEY FINDINGS')
print('=' * 60)
print()
print(f'  Dataset overview')
print(f'    Total transactions : {n_transactions:,}')
print(f'    Unique items       : {n_unique_items:,}')
print(f'    Dataset sparsity   : {sparsity:.2%}')
print()
print(f'  Apriori results')
print(f'    Frequent itemsets  : {len(frequent_itemsets):,}')
print(f'    Association rules  : {len(basket_rules):,}')
n_sig = basket_rules['significant'].sum()
print(f'    Significant rules  : {n_sig} ({n_sig/len(basket_rules)*100:.1f}%)')
print()
best_rule = basket_rules.sort_values('lift', ascending=False).iloc[0]
ant_str = ', '.join(list(best_rule['antecedents']))
con_str = ', '.join(list(best_rule['consequents']))
print(f'  Best rule by lift')
print(f'    {ant_str} → {con_str}')
print(f'    Support    = {best_rule["support"]:.4f}')
print(f'    Confidence = {best_rule["confidence"]:.4f}')
print(f'    Lift       = {best_rule["lift"]:.4f}')
print(f'    Kulczynski = {best_rule["kulczynski"]:.4f}')
print()
top_revenue_idx = bi_rules['estimated_revenue_impact'].idxmax()
top_rev = bi_rules.loc[top_revenue_idx]
print(f'  Top revenue opportunity')
print(f'    Rule : {top_rev["rule_label"]}')
print(f'    Est. revenue impact : ${top_rev["estimated_revenue_impact"]:,.2f}')
print()
print('  Actionable recommendations')
print('    1. Prioritise the top-lift significant rules for cross-selling campaigns')
print('    2. Use bundle candidates for promotional pricing')
print('    3. Apply store-layout hints to increase basket size')
print('    4. Re-run analysis periodically to detect seasonal shifts')
print('=' * 60)
logger.info('Analysis complete')
